> **用途**：个人知识点 Q&A（非交付文档）。  
# 02 数据处理 — 数据质量问题说明

> 记录时间：2026-05-15  
> 数据来源：01 旧 jsonl 上的诊断；02 已用 `parse_pmc.py` 重生成（见 schedule 阶段 3 进度）。  
> 发现方式：`med-data-EDA.ipynb` §0 / §2 跑通后的全量诊断

本文说明当前样本中已确认的**数据质量问题**：成因、影响、修复方式与推荐方案。


## ✅ 更新（2026-05-15）：已在 02 工程内修复

**结论**：问题来自 01 的 `parse_pmc_xml()` XPath，而非 PMC 源数据本身。02 已实现独立数据流，**全量期无需回 01**。

| 模块 | 路径 | 作用 |
|---|---|---|
| 解析器 | `02 数据处理/src/parse_pmc.py` | 修正 title / pub_year / pub_date / pmid / abstract |
| 批量构建 | `02 数据处理/src/build_jsonl.py` | XML 目录 → jsonl |
| 入口脚本 | `02 数据处理/scripts/build_jsonl.sh` | 命令行一键重跑 |

```bash
cd "02 数据处理"
./scripts/build_jsonl.sh --pmcids-from data/processed/sample.jsonl.bak01
```

**重跑后（100 篇）**：title>500 字符 **0 篇**；pub_year 正常；已含 `pmid`、`pub_date`；abstract 仍缺 3 篇（§3 定策略）。

下文「问题 1~4」保留为**历史记录**（01 旧 jsonl 上的诊断）。

## 问题总览

| # | 问题 | 规模（100 篇） | 严重程度 | 是否阻塞 §3 |
|---|---|---|---|---|
| 1 | `title` 被参考文献标题污染 | 96 篇 title > 500 字符 | **高** | 否（§3 应记录） |
| 2 | `pub_year` 多年份拼接 | 85 篇格式异常 | **中** | 否 |
| 3 | `abstract` 缺失 | 3 篇（3%） | **中** | 否（§3 定策略） |
| 4 | `pmid` 未抽取 | 100 篇均无 | **低**（任务书可选） | 否 |

**结论**：`load_pipeline` / notebook **无逻辑 bug**；问题来自 01 阶段 `parse_pmc_xml()` 的 XPath 设计。建议在 **§5 token 统计前** 修 parser 并重生成 jsonl；§3 可先基于现状做完整字段表并记录问题。

---

## 问题 1：`title` 异常过长、混入多篇文献标题

### 现象

- `title_char_len`：均值约 **4610** 字符，**96/100** 篇 > 500 字符，最长约 **21363** 字符。
- 正常 PMC 论文标题通常 < 300 字符。
- notebook 预览中 `retrieval_text` 开头为正确标题，后面紧接 `"Primary Ciliary Dyskinesia"`、`"Laterality defects..."` 等——实为**参考文献列表里的 `<article-title>`**。

### 原因

01 中 `parse_pmc_xml()` 使用：

```python
title = _text("//article-title")  # 匹配全文任意位置的 article-title
```

PMC 使用 **JATS XML**。除 `<front>` 里本篇标题外，`<back><ref-list>` 中每条参考文献也有 `<article-title>`（被引文献标题）。

XPath `//article-title` 会选中**所有**节点；`_text()` 用空格把它们**拼成一条字符串**，导致 `title` 字段变成「本篇标题 + 数十篇参考文献标题」。

01 notebook §5.4 打印 XML 前 80 行时已可见：第 5 行是正确标题，第 9、14、27 行等是 ref 中的 `<article-title>`——与污染现象一致。

### 影响

| 环节 | 影响 |
|---|---|
| **检索单元** `title + abstract` | `retrieval_text` 前半段是噪声，embedding 被无关标题稀释，检索准确率下降 |
| **Token 长度统计（§5）** | 若对 `retrieval_text` 计 token，P95/P99 严重偏大，**无法**据此定 chunk 策略 |
| **字段完整性（§3）** | `title` 非空率 100%，但**语义上不可用**，需在报告中区分「有值」与「可信」 |
| **说明文档** | 若不注明，易误判为「医学论文标题都特别长」 |

### 如何修复

**根因修复**：只取 `<front>` 内、本篇元数据下的标题，例如：

```python
# 推荐：限定 front，取第一个 title-group 下的 article-title
title = _text("//front//title-group/article-title[1]")
# 或更窄：
# title = _text("(//front//article-meta//title-group/article-title)[1]")
```

**验证**：对单篇 XML（如 `PMC8774754.xml`）解析后，`len(title)` 应在几十～三百字符量级；与改前数千字符对比。

**重跑**：用修正后的 `parse_pmc_xml()` 对 `01 验证模型/data/raw/extracted/`（或复制到 02 的 raw）批量重生成 `sample.jsonl`，再复制/覆盖到 `02 数据处理/data/processed/`。

### 推荐方案（优先）

1. 在 `02 数据处理/src/parse_pmc.py` 实现修正版 parser（从 01 迁入并改 XPath）。
2. 对现有 284 篇 XML 重跑 → 更新 `sample.jsonl`（100 篇子集或全量）。
3. 重新执行 notebook §2，确认 `title_char_len` 中位数回落到合理范围。
4. **§5 及以后**再基于 `title + abstract` 做 token 与分割设计。

**临时绕行**（若暂不重跑）：§5 仅对 **`abstract` 单独**做 token 分布；`title` 在文档中标注「待 parser 修复」。

---

## 问题 2：`pub_year` 出现 `"2022 2022"`、`"2023 2023 2023"` 等

### 现象

- **85/100** 条 `pub_year` 含重复或多余年份字符串。
- 示例：`2022 2022`、`2023 2023 2023`、`2023 2026 2023`。

### 原因

01 中使用：

```python
pub_year = _text("//pub-date/year")
```

JATS 中 `<pub-date>` 可出现多次：

- `<front>` 内：出版日期、电子版日期、收稿/修回日期等；
- `<ref-list>` 内：每条参考文献自己的出版年。

与 `title` 相同，全局 XPath 把所有 `<year>` **拼接**成一条，产生重复年份。

### 影响

| 环节 | 影响 |
|---|---|
| **「近 5 年」过滤** | 无法直接 `pub_year >= 2021`；需清洗或重抽 |
| **按年聚合统计** | 年份分布图失真 |
| **元数据展示** | 用户可见字段不规范 |

任务书字段为 `pub_date`，当前仅有 `pub_year` 且质量差——说明文档中需写清**字段映射与限制**。

### 如何修复

```python
# 推荐：只取 front 内文章级 pub-date 的第一个 year
pub_year = _text("//front//pub-date/year[1]")

# 更严谨：优先 epub / ppub
# pub_year = _text("//front//pub-date[@pub-type='epub']/year") or _text("//front//pub-date[@pub-type='ppub']/year")
```

若任务书要求完整 `pub_date`，可从同一 `<pub-date>` 节点组 `year/month/day` 拼 ISO 日期字符串，字段名 `pub_date`。

### 推荐方案

与问题 1 **同一轮 parser 修复**一并改 XPath，重跑 jsonl。§3 中对 `pub_year` 增加「清洗后唯一年份」检查（正则取第一个四位年或解析后整数）。

---

## 问题 3：`abstract` 缺失 3%（3/100 篇）

### 现象

| pmcid | journal |
|---|---|
| PMC12110669 | Current Issues in Molecular Biology |
| PMC12882845 | Metabolic Brain Disease |
| PMC12882890 | Nuclear Medicine and Molecular Imaging |

- 缺失率 **3%** > 任务书阈值 **1%**，触发清洗策略决策。
- `title`、`body` 仍存在；`has_abstract=False`。

### 原因

可能两类（需对单篇 XML 抽检确认）：

1. **源 XML 无结构化 abstract**（仅正文、或 abstract 非 `<abstract><p>` 结构），`//abstract//p` 抽不到文本。
2. **abstract 在其它标签内**（如 `trans-abstract`、`abstract[@abstract-type=...]`），当前 XPath 过窄。

这与 XPath 全局污染不同，属于**覆盖率**问题；若 XML 确实无摘要，则属正常缺失。

### 影响

| 环节 | 影响 |
|---|---|
| **RAG 检索块** | 任务书默认以摘要为检索单元；无 abstract 则无法形成有效 chunk |
| **缺失率指标** | 全量库若比例类似，清洗规则必须写进说明文档 |
| **样本量** | 丢弃后验证集为 97 篇，对 100 篇验证影响小 |

### 如何修复 / 处理

**策略 A（推荐，RAG 场景）**：pipeline 增加过滤 `has_abstract == True`，丢弃 3 篇；说明文档记录丢弃规则与数量。

**策略 B**：填充占位符 `"[NO ABSTRACT]"`——不推荐，会污染向量检索。

**策略 C**：放宽 XPath，例如 `//front//abstract//p` 或合并多种 abstract 类型后再判空；若仍空再丢弃。

### 推荐方案

1. §3 对 3 篇各打开 XML 确认是「真无摘要」还是「XPath 漏抽」。
2. 若为漏抽 → 改 parser；若为真无 → **采用策略 A 丢弃**。
3. 在 `load_pipeline` 或下游增加 `drop_missing_abstract=True` 可选参数，并写入 `sample_clean.jsonl`。

---

## 问题 4：`pmid` 未抽取（任务书字段缺失）

### 现象

- jsonl 中无 `pmid` 列；无法生成 `https://pubmed.ncbi.nlm.nih.gov/{pmid}/` 追溯链接。

### 原因

01 的 `parse_pmc_xml()` 未抽取：

```python
pmid = _text("//article-id[@pub-id-type='pmid']")
```

多数 PMC 全文 XML 在 `<front><article-meta>` 下含 `pub-id-type="pmid"` 的 `<article-id>`，属**功能遗漏**而非 XPath 污染。

### 影响

- 任务书「pmid 能否追溯原文」：**当前不能**。
- 可用 `pmcid` 链 PMC：`https://www.ncbi.nlm.nih.gov/pmc/articles/{PMCID}/` 作为替代。

### 推荐方案

与问题 1/2 **同轮 parser 修复**增加 `pmid` 字段；无 pmid 的篇保留空字符串并在 §3 统计覆盖率。

---

## 修复实施顺序（推荐）

```mermaid
flowchart LR
  A[修正 parse_pmc.py XPath] --> B[对 XML 批量重跑]
  B --> C[更新 sample.jsonl]
  C --> D[重跑 notebook §2 验证]
  D --> E[§3 字段完整性]
  E --> F[§5 token 用 title+abstract]
```

| 步骤 | 动作 | 产出 |
|---|---|---|
| 1 | `02/src/parse_pmc.py`：title / pub_year / pmid / abstract XPath | 可复用函数 |
| 2 | 输入：`01/data/raw/extracted/**/*.xml` | — |
| 3 | 输出：`02/data/processed/sample.jsonl`（及可选 `sample_clean.jsonl`） | 清洗后 97~100 篇 |
| 4 | `med-data-EDA.ipynb` §2 复跑 | title 长度正常、pub_year 单年 |
| 5 | 继续 §3~§6 | 说明文档与图表 |

**若时间紧**：先完成 §3（基于脏数据如实记录问题 + 清洗决策），再集中半天修 parser 并重跑；§5 前必须完成重跑或改用「仅 abstract」统计。

---

## 附录：诊断命令备忘

在 `02 数据处理` 根目录、环境 `med-rag-verify` 下：

```bash
cd "02 数据处理"
python -c "
import sys; sys.path.insert(0,'src')
from load_pipeline import resolve_project_dir, setup_paths, load_pmc_jsonl
pd,_=resolve_project_dir(); p=setup_paths(pd)
ds=load_pmc_jsonl(p['sample_jsonl'], backend='pandas', add_derived=True)
print('title>500:', (ds.title_char_len>500).sum(), '/', len(ds))
print('abstract缺失:', (~ds.has_abstract).sum())
"
```

相关文件：

- 计划：`02 数据处理/schedule.md` → 「数据质量问题」小节
- 分析：`02 数据处理/notebooks/med-data-EDA.ipynb`
- 01 原 parser：`01 验证模型/med-LLM-RAG.ipynb` §5.5

---

## 补充 Q&A

> 以下是对「数据问题最终怎么解决」「`src/` 各文件作用」「相对 01 继承/修改」「外接盘要设什么」的整理，便于与上文**问题 1~4 的历史诊断**对照回顾。

### Q1：针对之前的数据问题，最后采用什么方法解决？

分两层：

1. **根因（01 旧解析 XPath）** — title / pub_year 被全局 XPath 拼进参考文献；pmid 未抽。  
   **做法**：在 **`02 数据处理/src/parse_pmc.py`** 重写 XPath（限定 `//front//...`），用 **`build_jsonl.py`** + **`scripts/build_jsonl.sh`** 从 **XML 重新批量生成** `sample.jsonl`，不再依赖 01 notebook 里那段旧 `parse_pmc_xml()`。

2. **业务规则（非 parser bug）** — 约 **3%** 篇 XML 仍无可用 abstract。  
   **做法**：在 **`load_pipeline.py`** + notebook **§3** 中 **丢弃无 abstract**，得到 **`sample_clean.jsonl`（97 篇）** 供后续分析。

**一句话**：抽取规则在 **`parse_pmc.py`** 修；数据文件用 **`build_jsonl`** 重造；无摘要用 **清洗策略丢弃**。

---

### Q2：`02 数据处理/src/` 里各文件什么作用？

| 文件 | 作用 |
|---|---|
| **`parse_pmc.py`** | 单篇 PMC **JATS XML → dict**（title/abstract/body/journal/pub_year/pub_date/pmid/字符数）。**数据抽取质量的核心。** |
| **`build_jsonl.py`** | **CLI**：递归扫 XML 目录，调用 `parse_pmc_xml`，写 **JSONL**；支持 `--limit`、`--pmcids-from`、自动探测 XML 根目录。 |
| **`load_pipeline.py`** | **读 jsonl 做分析**：路径、`MED_RAG_DATA_ROOT`、`datasets`/`pandas`、派生列、§3 完整性/质量/元数据、`drop_missing_abstract` 等。**不负责从 XML 抽字段。** |
| **`domain_analysis.py`** | **阶段 4**（领域理解）骨架：abstract token 分层、IMRaD/缩写等；与「修脏数据」无关。 |

---

### Q3：解决「数据问题」主要在哪个代码文件？

- **改抽取逻辑**（title/pub_year/pmid 等）：**`parse_pmc.py`**。  
- **把修复落地成新 jsonl**：**`build_jsonl.py`**（`build_jsonl.sh` 只是调 Python）。  
- **`load_pipeline.py`** 只消费已生成好的 jsonl；若 jsonl 仍是 01 旧版，它无法消除 title 污染。

---

### Q4：相对 01 的数据处理，继承了什么、修改了什么？

**继承（思想）**

- 仍是 **lxml + XPath** 读 JATS，输出 **一行一篇 jsonl**。
- 字段与 01 目标一致，02 额外 **`pmid`、`pub_date`**。

**修改（相对 01 `med-LLM-RAG.ipynb` §5.5）**

| 点 | 01 | 02 |
|---|---|---|
| title | `//article-title` | `//front//title-group/article-title[1]` |
| pub_year | `//pub-date/year` 全局 | `front` 内 epub/ppub 等，取单年 |
| pmid | 无 | 有 |
| pub_date | 无 | 有 |
| 形态 | 写在 ipynb cell | **独立 `.py` + 可脚本批跑** |

---

### Q5：未来接外接盘全量数据，要改 / 设置什么？

**环境变量（优先，少改代码）**

- **`MED_RAG_DATA_ROOT`**：外接盘数据根，如 `/Volumes/盘名/med-rag-pmc`（其下 `extracted/`、`processed/` 与 schedule 约定一致）。
- **`PMC_XML_ROOT`**（可选）：若 XML 根不在默认推断路径，单独指定解压目录。

**命令示例**

```bash
export MED_RAG_DATA_ROOT=/Volumes/你的盘/med-rag-pmc
export PMC_XML_ROOT=/Volumes/你的盘/med-rag-pmc/extracted   # 需要时
cd "02 数据处理"
./scripts/build_jsonl.sh -o data/processed/oa_comm_full.jsonl
```

**代码里已预留**

- `build_jsonl.resolve_xml_root()`：按 `PMC_XML_ROOT` → `MED_RAG_DATA_ROOT/extracted` → 本工程 `data/raw/extracted` → 01 extracted 顺序探测。
- `load_pipeline.setup_paths()`：若设 `MED_RAG_DATA_ROOT`，读 jsonl 会切到该根下的 `processed/`。

**一般不必改 `parse_pmc.py`**，除非全量 XML 与当前子集结构差异大再微调 XPath。

---

*与 `schedule.md` 数据流、阶段 B 一致；以仓库当前代码为准。*